In [1]:
from io import StringIO

import pandas as pd
import requests
import logging
logger = logging.getLogger(__name__)


def get_data_from_ilo_for_one_indicator(indicator_id= "EMP_5EMP_SEX_OC2_NB_Q",
                                        from_date: int = 2014,
                                        to_date: int = 2026) -> pd.DataFrame | None:
    """
    Récupère les données de l'API ILO pour un indicateur donné sur une période spécifiée.
    Args:
        indicator_id: identifiant de l'indicateur à récupérer.
        from_date: année de début de la période.
        to_date: année de fin de la période.
    Returns:
        pd.DataFrame | None: DataFrame contenant les données, ou None en cas d'erreur.
    """
       
    logger.info(f"Début de la fonction get_data_from_ilo_for_one_indicator pour l'indicateur {indicator_id} de {from_date} à {to_date}")
    try:
        logger.info(f"Téléchargement des données depuis {from_date} jusqu'à {to_date}...")
        r = requests.get(f"https://rplumber.ilo.org/data/indicator?id={indicator_id}&timefrom={from_date}&timeto={to_date}&type=label&format=.csv")
        r.raise_for_status()
        logger.info("Requête réussie, lecture du CSV...")
        df = pd.read_csv(StringIO(r.text))
        logger.info("Affichage des premières lignes du DataFrame :")
        logger.info("Données récupérées avec succès.")
        return df
    except Exception as e:
        logger.error("Une erreur est survenue lors de la récupération des données:")
        logger.error(e)
        return None


In [37]:
df = get_data_from_ilo_for_one_indicator()

C:\Users\Salim\AppData\Local\Temp\ipykernel_43152\3055529138.py:28: DtypeWarning: Columns (0: note_classif.label) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(StringIO(r.text))


In [43]:
df.head()

,ref_area.label,source.label,indicator.label,sex.label,classif1.label,time,obs_value,obs_status.label,note_classif.label,note_indicator.label,note_source.label,ISCO,digit_level,occupation
0,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: Total",2025Q4,8876.650,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,NaN,NaN,NaN
1,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: 01 - Comm...",2025Q4,30.482,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Occupation (ISCO-08),01,Commissioned armed forces officers
2,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: 02 - Non-...",2025Q4,78.433,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Occupation (ISCO-08),02,Non-commissioned armed forces officers
3,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: 03 - Arme...",2025Q4,19.609,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Occupation (ISCO-08),03,"Armed forces occupations, other ranks"
4,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: 11 - Chie...",2025Q4,24.500,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Occupation (ISCO-08),11,"Chief executives, senior officials and legisla..."


In [40]:
import re

def process_classif1_label_column(df):
    """
    Ajoute les colonnes 'ISCO', 'digit_level' et 'occupation' au DataFrame en analysant 'classif1.label'.
    Extrait le code socioprofessionnel à 2 chiffres INSEE dans digit_level.
    """
    def split_lbas(val):
        # val attendu du type : "Occupation (ISCO-08), 2 digit level: 01 - Commissioned armed forces officers"
        if pd.isnull(val):
            return pd.Series([None, None, None])
        pattern = r'^([^,]+),\s*([^:]+:\s*\d{2})\s*-\s*(.*)$'
        match = re.match(pattern, str(val))
        if match:
            return pd.Series([match.group(1).strip(), match.group(2).strip(), match.group(3).strip()])
        else:
            return pd.Series([None, None, None])
    
    df[['ISCO', 'digit_level', 'occupation']] = df['classif1.label'].apply(split_lbas)
    df["digit_level"] = df['digit_level'].str.extract(r'(\d{2})') # code socioprofessionnel a 2 chiffres de INSEE
    return df

# Utilisation :
df = process_classif1_label_column(df)

In [ ]:
df.head()

,ref_area.label,source.label,indicator.label,sex.label,classif1.label,time,obs_value,obs_status.label,note_classif.label,note_indicator.label,note_source.label,ISCO,digit_level,occupation
0,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: Total",2025Q4,8876.650,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,NaN,NaN,NaN
1,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: 01 - Comm...",2025Q4,30.482,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Occupation (ISCO-08),01,Commissioned armed forces officers
2,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: 02 - Non-...",2025Q4,78.433,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Occupation (ISCO-08),02,Non-commissioned armed forces officers
3,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: 03 - Arme...",2025Q4,19.609,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Occupation (ISCO-08),03,"Armed forces occupations, other ranks"
4,Angola,LFS - Employment Survey,Employment by sex and occupation - ISCO level ...,Total,"Occupation (ISCO-08), 2 digit level: 11 - Chie...",2025Q4,24.500,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Occupation (ISCO-08),11,"Chief executives, senior officials and legisla..."


In [44]:
# Compter les occurrences de chaque occupation
occupation_counts = df['digit_level'].unique()

# Afficher les résultats
print(occupation_counts)

<StringArray>
[ nan, '01', '02', '03', '11', '12', '13', '14', '21', '22', '23', '24', '25',
 '26', '31', '32', '33', '34', '35', '41', '42', '43', '44', '51', '52', '53',
 '54', '61', '62', '63', '71', '72', '73', '74', '75', '81', '82', '83', '91',
 '92', '93', '94', '95', '96']
Length: 44, dtype: str


In [8]:
# Compter les occurrences de chaque occupation
occupation_counts = df['ISO'].value_counts()

# Afficher les résultats
print(occupation_counts)

ISO
Occupation (ISCO-08)    236919
Occupation (ISCO-88)      2571
Name: count, dtype: int64


In [9]:
# Compter les occurrences de chaque occupation
occupation_counts = df['ref_area.label'].value_counts()

# Afficher les résultats
print(occupation_counts)

ref_area.label
Portugal                                                6552
Brazil                                                  6468
Austria                                                 6267
United Kingdom of Great Britain and Northern Ireland    6258
Switzerland                                             6204
                                                        ... 
Pakistan                                                 498
Tanzania, United Republic of                             387
Barbados                                                 249
Thailand                                                 135
Gambia                                                   132
Name: count, Length: 65, dtype: int64


In [12]:
# Compter les occurrences de chaque occupation par combinaison de 'ref_area.label' et 'occupation'
occupation_counts = df.groupby(['ref_area.label', 'occupation']).size()

# Afficher les résultats
print(occupation_counts)

ref_area.label  occupation                                                 
Angola          Administrative and commercial managers                         60
                Agricultural, forestry and fishery labourers                   60
                Armed forces occupations, other ranks                          60
                Assemblers                                                     60
                Building and related trades workers, excluding electricians    60
                                                                               ..
Zimbabwe        Science and engineering professionals                          39
                Stationary plant and machine operators                         39
                Street and related sales and service workers                   39
                Subsistence farmers, fishers, hunters and gatherers            39
                Teaching professionals                                         39
Length: 2684, dtype: i

In [33]:
df2 = get_data_from_ilo_for_one_indicator(indicator_id="EMP_5WAP_SEX_AGE_RT_Q", from_date=2014, to_date=2026)


In [28]:
df2.head()

,ref_area.label,source.label,indicator.label,sex.label,classif1.label,time,obs_value,obs_status.label,note_classif.label,note_indicator.label,note_source.label
0,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Total,"Age (Youth, adults): 15+",2025Q4,39.584,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...
1,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Total,"Age (Youth, adults): 15-24",2025Q4,17.948,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...
2,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Total,"Age (Youth, adults): 25+",2025Q4,51.850,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...
3,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Male,"Age (Youth, adults): 15+",2025Q4,43.283,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...
4,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Male,"Age (Youth, adults): 15-24",2025Q4,18.174,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...


In [ ]:
df2.head()

,ref_area.label,source.label,indicator.label,sex.label,classif1.label,time,obs_value,obs_status.label,note_classif.label,note_indicator.label,note_source.label,ISCO,digit_level,occupation
0,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Total,"Age (Youth, adults): 15+",2025Q4,39.584,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,NaN,NaN,NaN
1,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Total,"Age (Youth, adults): 15-24",2025Q4,17.948,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Age (Youth,15,24
2,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Total,"Age (Youth, adults): 25+",2025Q4,51.850,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,NaN,NaN,NaN
3,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Male,"Age (Youth, adults): 15+",2025Q4,43.283,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,NaN,NaN,NaN
4,Angola,LFS - Employment Survey,Employment-to-population ratio by sex and age ...,Male,"Age (Youth, adults): 15-24",2025Q4,18.174,NaN,NaN,Frequency: Quarterly,Repository: ILO-STATISTICS - Micro data proces...,Age (Youth,15,24


In [ ]:
# Compter les occurrences de chaque occupation
occupation_counts = df2.groupby(['classif1.label', 'sex.label','ref_area.label']).size()

# Afficher les résultats
print(occupation_counts)

classif1.label              sex.label
Age (Youth, adults): 15+    Female       2402
                            Male         2402
                            Other           8
                            Total        2402
Age (Youth, adults): 15-24  Female       2402
                            Male         2402
                            Other           4
                            Total        2402
Age (Youth, adults): 25+    Female       2402
                            Male         2402
                            Other           7
                            Total        2402
dtype: int64


In [48]:
df2.columns

Index(['ref_area.label', 'source.label', 'indicator.label', 'sex.label',
       'classif1.label', 'time', 'obs_value', 'obs_status.label',
       'note_classif.label', 'note_indicator.label', 'note_source.label'],
      dtype='str')